In [ ]:
# STEP 1: Install libraries
!pip install librosa soundfile tensorflow scikit-learn tqdm
# STEP 2: Download + unzip dataset
!wget -q –show-process
https://zenodo.org/record/1188976/files/Audio_Speech_Actors_o1-24.zip
!unzip -o Audio_Speech_Actors_01-24.zip -d ./ravdess/
# STEP 3: Imports
import os, pickle, warnings
import numpy as np
import soundfile
import librosa
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
# STEP 4: Auto-detect dataset path
dataset_path = None
for root, dirs, files in os.walk("./ravdess/"):
    for d in dirs:
        if d.startswith("Actor_"):
            dataset_path = root
            break
    if dataset_path:
        break
if dataset_path is None:
    raise FileNotFoundError("Could not find Actor_* folders in ./ravdess/")
else:
    print(f"Dataset path automatically detected: {dataset_path}")
# STEP 5.5: Normalize all audio durations (e.g., 3 seconds each)
import soundfile as sf
fixed_duration = 4.0   # target duration in seconds
target_sr = 22050      # target sample rate
output_dir = "./ravdess_normalized/"
print("Normalizing all audio files to same duration...")
os.makedirs(output_dir, exist_ok=True)
actor_folders = [f for f in os.listdir(dataset_path) if f.startswith("Actor_")]
for actor in tqdm(actor_folders, desc="Actors"):
    actor_path = os.path.join(dataset_path, actor)
    new_actor_path = os.path.join(output_dir, actor)
    os.makedirs(new_actor_path, exist_ok=True)
    for file in os.listdir(actor_path):
        if file.endswith(".wav"):
            src = os.path.join(actor_path, file)
            dst = os.path.join(new_actor_path, file)
            try:
                y, sr = librosa.load(src, sr=target_sr)
                desired_length = int(fixed_duration * sr)
                if len(y) < desired_length:
                    y = np.pad(y, (0, desired_length - len(y)), mode='constant')
                elif len(y) > desired_length:
                    y = y[:desired_length]
                sf.write(dst, y, sr)
            except Exception as e:
                print(f"Error processing {src}: {e}")
print("All audio files normalized to same duration!")
# Use normalized dataset for next steps
dataset_path = output_dir
# STEP 5: Feature extractor (MFCC + Chroma + Mel + Contrast)
def extract_feature(file_name, n_mfcc=40):
    try:
        with soundfile.SoundFile(file_name) as sound:
            X = sound.read(dtype="float32")
            sr = sound.samplerate
            if len(X) < 2048:
                return None
            mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sr, n_mfcc=n_mfcc).T, axis=0)
            chroma = np.mean(librosa.feature.chroma_stft(y=X, sr=sr).T, axis=0)
            mel = np.mean(librosa.feature.melspectrogram(y=X, sr=sr).T, axis=0)
            contrast = np.mean(librosa.feature.spectral_contrast(y=X, sr=sr).T, axis=0)
            feature = np.hstack([mfccs, chroma, mel, contrast])
            feature = np.array(feature).flatten().astype(np.float32)
            return feature
    except:
        return None
# STEP 6: Load dataset with caching
emotions = {
    '01':'neutral', '02':'calm', '03':'happy', '04':'sad',
    '05':'angry', '06':'fearful', '07':'disgust', '08':'surprised'
}
if os.path.exists("features_full.pkl"):
    print("Loading cached features...")
    with open("features_full.pkl", "rb") as f:
        X, y = pickle.load(f)
else:
    print("Extracting features (may take several minutes)...")
    X, y = [], []
    actor_folders = [f for f in os.listdir(dataset_path) if f.startswith("Actor_")]
    for actor in tqdm(actor_folders, desc="Actors"):
        actor_path = os.path.join(dataset_path, actor)
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):
                emotion_code = file.split('-')[2]
                if emotion_code in emotions:
                    feature = extract_feature(os.path.join(actor_path, file))
                    if feature is not None and len(feature) > 0:
                        X.append(feature)
                        y.append(emotions[emotion_code])
    with open("features_full.pkl", "wb") as f:
        pickle.dump((X, y), f)
    print("Features cached!")
print(f"Total usable files: {len(X)}")
import os
from collections import Counter
import matplotlib.pyplot as plt
#Get emotion labels safely from filenames
dataset_path = "./ravdess/"
emotions = {
    '01':'neutral', '02':'calm', '03':'happy', '04':'sad',
    '05':'angry', '06':'fearful', '07':'disgust', '08':'surprised'
}
emotion_labels = []
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".wav"):
            emotion_code = file.split('-')[2]
            if emotion_code in emotions:
                emotion_labels.append(emotions[emotion_code])
# Count occurrences
emotion_counts = Counter(emotion_labels)
#Bar chart
plt.figure(figsize=(8,5))
colors = ['skyblue','lightgreen','salmon','orange','pink','purple','yellow','lightcoral']
plt.bar(emotion_counts.keys(),emotion_counts.values(), color=colors[:len(emotion_counts)])
plt.title("Number of Audio Files per Emotion")
plt.xlabel("Emotion")
plt.ylabel("Number of Files")
plt.xticks(rotation=45)
for i, v in enumerate(emotion_counts.values()):
    plt.text(i, v+2, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
# STEP 7: Pad features & normalize
max_len = max(len(f) for f in X)
X_padded = []
for f in X:
    if len(f) < max_len:
        f = np.pad(f, (0, max_len - len(f)), mode='constant')
    elif len(f) > max_len:
        f = f[:max_len]
    X_padded.append(f)
X = np.stack(X_padded)
scaler = StandardScaler()
X = scaler.fit_transform(X)
le = LabelEncoder()
y = to_categorical(le.fit_transform(y))
# STEP 8: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# STEP 9: Build neural network
model = Sequential()
model.add(Dense(512, input_shape=(X.shape[1],), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(y.shape[1], activation='softmax'))
model.compile(loss='categorical_crossentropy',optimizer='adam', metrics=['accuracy'])
model.summary()

# STEP 10: Train model
early_stop=EarlyStopping(monitor='val_accuracy',patience=10, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    batch_size=32,
    epochs=100,
    callbacks=[early_stop],
    verbose=1
)
# STEP 11: Evaluate
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc*100:.2f}%")
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
# Step 1: Predict on the test set
y_pred_probs = model.predict(X_test)
y_pred_labels = np.argmax(y_pred_probs, axis=1)
y_true_labels = np.argmax(y_test, axis=1)
# Step 2: Generate confusion matrix
cm = confusion_matrix(y_true_labels, y_pred_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
# Step 3: Visualize
plt.figure(figsize=(8, 6))
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title("Deep Learning Model - Confusion Matrix")
plt.show()
# Optional: Print detailed classification report
print("Classification Report (Deep Learning):")
print(classification_report(y_true_labels, y_pred_labels, target_names=le.classes_))
# STEP 12: Predict frame-wise emotions
def extract_features_frame(y, sr, frame_duration=1.0, n_mfcc=40):
    frame_length = int(sr * frame_duration)
    hop_length = frame_length
    frames = []
    for start in range(0, len(y), hop_length):
        frame = y[start:start + frame_length]
        if len(frame) < 2048:
            continue
        mfccs = np.mean(librosa.feature.mfcc(y=frame, sr=sr, n_mfcc=n_mfcc).T, axis=0)
        chroma = np.mean(librosa.feature.chroma_stft(y=frame, sr=sr).T, axis=0)
        mel = np.mean(librosa.feature.melspectrogram(y=frame, sr=sr).T, axis=0)
        contrast = np.mean(librosa.feature.spectral_contrast(y=frame, sr=sr).T, axis=0)
        feature = np.hstack([mfccs, chroma, mel, contrast])
        feature = np.array(feature).flatten().astype(np.float32)
        if len(feature) < X.shape[1]:
            feature = np.pad(feature, (0, X.shape[1] - len(feature)), mode='constant')
        elif len(feature) > X.shape[1]:
            feature = feature[:X.shape[1]]
        frames.append(feature)
    return np.stack(frames)
def predict_emotions_over_time(file, frame_duration=1.0):
    y, sr = librosa.load(file, sr=None)
    frames = extract_features_frame(y, sr, frame_duration=frame_duration)
    frames_scaled = scaler.transform(frames)
    preds = model.predict(frames_scaled)
    pred_emotions = [le.inverse_transform([np.argmax(p)])[0] for p in preds]
    emotion_seconds = {emotion: 0 for emotion in emotions.values()}
    for e in pred_emotions:
        emotion_seconds[e] += frame_duration
    return pred_emotions, emotion_seconds, y, sr
# STEP 13: Upload audio
from google.colab import files
from IPython.display import Audio
uploaded = files.upload()
filename = list(uploaded.keys())[0]
display(Audio(filename))
import seaborn as sns
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
# Define thresholds per emotion (seconds)
emotion_thresholds = {
    'neutral': 2.0,
    'calm': 2.0,
    'happy': 2.0,
    'sad': 1.5,
    'angry': 2.5,
    'fearful': 2.0,
    'disgust': 2.0,
    'surprised': 1.5
}
# Define sentiment groups
sentiment_map = {
    'happy': 'Positive',
    'surprised': 'Positive',
    'calm': 'Neutral',
    'neutral': 'Neutral',
    'sad': 'Negative',
    'angry': 'Negative',
    'fearful': 'Negative',
    'disgust': 'Negative'
}
# Emotion color palette
color_map = {
    'neutral': 'gray', 'calm': 'lightblue', 'happy': 'gold',
    'sad': 'blue', 'angry': 'red', 'fearful': 'purple',
    'disgust': 'green', 'surprised': 'orange'
}
def compute_overall_sentiment(emotion_seconds):
    """Determine the overall sentiment based on emotion durations and thresholds."""
    satisfied = [e for e, dur in emotion_seconds.items() if dur >= emotion_thresholds[e]]
    if satisfied:
        main_emotion = max(satisfied, key=lambda e: emotion_seconds[e])
    else:
        main_emotion = max(emotion_seconds, key=emotion_seconds.get)
    overall_sentiment = sentiment_map.get(main_emotion, "Neutral")
    return main_emotion, overall_sentiment
def visualize_audio_emotions(file):
    """Predict emotions over time and visualize waveform, emotion timeline, and sentiment."""
    pred_emotions, emotion_seconds, y, sr = predict_emotions_over_time(file)
    # Compute final emotion & sentiment
    main_emotion, overall_sentiment = compute_overall_sentiment(emotion_seconds)
    # Print summary
    print(f"\n Audio Sentiment Analysis Results")
    print("="*45)
    print(f"Dominant Emotion: {main_emotion.upper()}")
    print(f"Overall Sentiment: {overall_sentiment.upper()}\n")
    print("Duration per Emotion:")
    for e, s in emotion_seconds.items():
        print(f"  • {e:10s}: {s:.1f} sec (threshold={emotion_thresholds[e]}s)")
    # Pitch Extraction
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    pitch_values = np.array([
        pitches[:, i][magnitudes[:, i].argmax()] for i in range(pitches.shape[1])
    ])
    pitch_values[pitch_values == 0] = np.nan
    # --- Visualization ---
    plt.figure(figsize=(14, 12))
    plt.suptitle(
        f"Audio Emotion & Sentiment Analysis\nMain Emotion: {main_emotion.upper()} | Sentiment: {overall_sentiment.upper()}",
        fontsize=16, fontweight='bold', color='purple'
    )
    # Mel-Spectrogram Representation
    plt.subplot(4, 1, 1)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel', cmap='magma')
    plt.title("Mel-Spectrogram Representation of Speech Sample", fontsize=13)
    plt.xlabel("Time (s)")
    plt.ylabel("Mel Frequency (Hz)")
    plt.colorbar(format="%+2.0f dB")
    # Waveform
    plt.subplot(4, 1, 2)
    librosa.display.waveshow(y, sr=sr, color='steelblue')
    plt.title("Waveform", fontsize=13)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    # Emotion Timeline (1 bar = 1 second)
    plt.subplot(4, 1, 3)
    times = np.arange(len(pred_emotions))
    colors = [color_map[e] for e in pred_emotions]
    plt.bar(times, [1]*len(times), color=colors, edgecolor='none', width=1.0)
    plt.yticks([])
    plt.title("Emotion Timeline (1 bar = 1 second)", fontsize=13)
    plt.xlabel("Time (seconds)")
    legend_patches = [plt.Line2D([0], [0], color=color_map[e], lw=6, label=e)
                      for e in color_map]
    plt.legend(handles=legend_patches, bbox_to_anchor=(1.05, 1), loc='upper left')
    # Emotion Duration Distribution
    plt.subplot(4, 1, 4)
    sns.barplot(x=list(emotion_seconds.keys()), y=list(emotion_seconds.values()),
                palette=color_map)
    plt.title("Emotion Duration Distribution", fontsize=13)
    plt.xlabel("Emotion")
    plt.ylabel("Duration (seconds)")
    plt.xticks(rotation=30)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
# STEP 15: Run visualization
visualize_audio_emotions(filename)
# Replace the previous "Compare Traditional ML Models" cell with this
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
# Ensure X_train, X_test, y_train, y_test, scaler, le, model (Keras) are already in memory
# Prepare flat arrays (they should already be numpy arrays)
X_train_flat = np.array(X_train)
X_test_flat = np.array(X_test)
# Ensure labels are integer class indices
if len(y_train.shape) > 1 and y_tra
in.shape[1] > 1:
    y_train_labels = np.argmax(y_train, axis=1)
    y_test_labels = np.argmax(y_test, axis=1)
else:
    y_train_labels = y_train.ravel()
    y_test_labels = y_test.ravel()
# Traditional ML models to evaluate
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42)
}
results = {}
for name, m in models.items():
    print(f"\n=== Training {name} ===")
    m.fit(X_train_flat, y_train_labels)
    y_pred = m.predict(X_test_flat)
    acc = accuracy_score(y_test_labels, y_pred)
    results[name] = acc
    print(f"{name} Accuracy: {acc:.4f}")
    print("Classification Report:")
    print(classification_report(y_test_labels, y_pred, target_names=le.classes_))
    # Confusion Matrix
    cm = confusion_matrix(y_test_labels, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    plt.figure(figsize=(6,5))
    disp.plot(cmap='Blues', xticks_rotation=45, ax=plt.gca())
    plt.title(f"{name} - Confusion Matrix")
    plt.tight_layout()
    plt.show()
# Include Deep Learning model in the comparison
print("\n=== Deep Learning (Keras) Evaluation ===")
# If you haven't evaluated recently, compute test loss & acc
try:
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
except Exception:
    # fallback: compute predictions then accuracy
    y_pred_probs = model.predict(X_test)
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    test_acc = accuracy_score(y_test_labels, y_pred_labels)
# Add deep learning result
results["Deep Learning"] = test_acc
# Print classification report and confusion matrix for deep model
y_pred_probs = model.predict(X_test)
y_pred_labels = np.argmax(y_pred_probs, axis=1)
print(f"Deep Learning Test Accuracy: {test_acc:.4f}")
print("Classification Report (Deep Learning):")
print(classification_report(y_test_labels, y_pred_labels, target_names=le.classes_))
cm = confusion_matrix(y_test_labels, y_pred_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
plt.figure(figsize=(6,5))
disp.plot(cmap='Blues', xticks_rotation=45, ax=plt.gca())
plt.title("Deep Learning - Confusion Matrix")
plt.tight_layout()
plt.show()
# Final accuracy comparison plot
plt.figure(figsize=(8,5))
names = list(results.keys())
accs = [results[n] for n in names]
bars = plt.bar(names, accs)
plt.ylim(0,1)
plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison (Test set)")
plt.xticks(rotation=30)
for i, v in enumerate(accs):
    plt.text(i, v + 0.01, f"{v:.3f}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
print("\nModel accuracies (test set):")
for k, v in results.items():
    print(f" - {k}: {v:.4f}")
